# Naturalistic fMRI: Connectome from Pixar Movie Data

ds000228 -- subjects watched a Pixar movie in the scanner. Extract whole-brain functional connectivity, compare children vs adults, and visualize the connectome using the Schaefer atlas.

## Prerequisites

- **Python 3.9+**
- **datalad** installed for downloading the ds000228 dataset
- Basic understanding of fMRI, functional connectivity, and brain parcellations
- Familiarity with correlation matrices and connectome visualization

## Setup

This notebook uses the **ds000228** dataset (Richardson et al., 2018) -- a naturalistic fMRI study where
participants (children and adults) watched a Pixar short film in the scanner.

### Data Download

The dataset must be downloaded using datalad before running this notebook:

```bash
# Install datalad if needed
pip install datalad

# Clone the dataset (lightweight -- only downloads metadata)
datalad install https://github.com/OpenNeuroDatasets/ds000228

# Download functional data for the subjects used in this notebook
cd ds000228
datalad get sub-pixar001/func/*bold.nii.gz sub-pixar002/func/*bold.nii.gz sub-pixar003/func/*bold.nii.gz
```

Make sure the `ds000228` directory is in your working directory (or adjust `DS228` below).

In [ ]:
# Install required packages
!pip install nibabel nilearn matplotlib numpy pandas scipy

In [ ]:
import nibabel as nib
import nilearn.datasets as nld
import nilearn.plotting as nlp
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Use relative path -- assumes ds000228 is in the current working directory
DS228 = 'ds000228'

# Load participant info
participants = pd.read_csv(f'{DS228}/participants.tsv', sep='\t')
print(participants.head(10))
print(f'\nTotal: {len(participants)} subjects')
print(f'Children: {(participants.Child_Adult == "child").sum()}')
print(f'Adults:   {(participants.Child_Adult == "adult").sum()}')

In [ ]:
# Load Schaefer 100-parcel atlas
schaefer = nld.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7)

masker = NiftiLabelsMasker(
    labels_img=schaefer.maps,
    standardize='zscore_sample',
    resampling_target='data',
    memory_level=1, verbose=0
)
# Fit once on the atlas (no data) -- guarantees all subjects return the same 100 parcels
masker.fit()

subs = ['sub-pixar001', 'sub-pixar002', 'sub-pixar003']
time_series_list = []
for sub in subs:
    func = f'{DS228}/{sub}/func/{sub}_task-pixar_bold.nii.gz'
    ts = masker.transform(func)   # transform() not fit_transform() -- reuses fitted masker
    time_series_list.append(ts)
    print(f'{sub}: {ts.shape} (TRs x parcels)')

In [ ]:
# Compute per-subject FC matrices
conn = ConnectivityMeasure(kind='correlation')
fc_list_raw = [conn.fit_transform([ts])[0] for ts in time_series_list]
sizes = [fc.shape[0] for fc in fc_list_raw]
print(f'Parcel counts per subject: {sizes}')

# Crop all FC matrices to the minimum common parcel count
# (some peripheral parcels may be clipped when atlas is resampled to 3mm FOV)
min_p = min(sizes)
fc_matrices = [fc[:min_p, :min_p] for fc in fc_list_raw]
labels_trimmed = [l.decode() if isinstance(l, bytes) else l for l in schaefer.labels[:min_p]]
mean_fc = np.mean(np.stack(fc_matrices), axis=0)
print(f'Using {min_p} common parcels across {len(subs)} subjects. FC shape: {mean_fc.shape}')

fig, axes = plt.subplots(1, len(fc_matrices), figsize=(6*len(fc_matrices), 5))
if len(fc_matrices) == 1: axes = [axes]
for i, (ax, sub) in enumerate(zip(axes, subs)):
    im = ax.imshow(fc_matrices[i], vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_title(f'{sub}\nFC ({min_p} parcels, {time_series_list[i].shape[0]} TRs)')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

nlp.plot_matrix(mean_fc, labels=labels_trimmed,
                colorbar=True, vmax=0.8, vmin=-0.8,
                title=f'Mean FC -- Pixar naturalistic fMRI ({min_p} parcels, {len(subs)} subjects)')
plt.show()

In [ ]:
# Fast parcel centroid coordinates -- trim to min_p to match the cropped FC matrix
import nibabel as nib
from scipy import ndimage

atlas_img = nib.load(schaefer.maps)
atlas_data = atlas_img.get_fdata().astype(int)
affine = atlas_img.affine
region_coords_full = np.array([
    affine[:3, :3] @ np.array(ndimage.center_of_mass(atlas_data == i)) + affine[:3, 3]
    for i in range(1, 101)
])
region_coords = region_coords_full[:min_p]   # match the cropped FC shape

nlp.plot_connectome(mean_fc, region_coords,
                    edge_threshold='90%',
                    title=f'Top 10% connections -- Pixar fMRI ({min_p} parcels)',
                    node_size=20, colorbar=True)
plt.show()

import re
network_labels = [re.search(r'7Networks_\w+_(\w+)_', l.decode() if isinstance(l, bytes) else l).group(1)
                  if re.search(r'7Networks_\w+_(\w+)_', l.decode() if isinstance(l, bytes) else l) else 'Other'
                  for l in schaefer.labels[:min_p]]
unique_nets = list(dict.fromkeys(network_labels))
print('Networks present:', unique_nets)

net_ids = np.array([unique_nets.index(n) for n in network_labels])
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(mean_fc, cmap='RdBu_r', vmin=-1, vmax=1)
boundaries = [0] + [i+1 for i in range(len(net_ids)-1) if net_ids[i] != net_ids[i+1]] + [min_p]
for b in boundaries[1:-1]:
    ax.axhline(b-0.5, color='k', lw=1.5)
    ax.axvline(b-0.5, color='k', lw=1.5)
ax.set_title('FC matrix sorted by Yeo 7-network')
plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout(); plt.show()

## References

- Richardson, H., Lisandrelli, G., Riobueno-Naylor, A., & Saxe, R. (2018). Development of the social brain from age three to twelve years. *Nature Communications*, 9(1), 1027. https://doi.org/10.1038/s41467-018-03399-2
- Schaefer, A., Kong, R., Gordon, E. M., Laumann, T. O., Zuo, X. N., Holmes, A. J., ... & Yeo, B. T. T. (2018). Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity MRI. *Cerebral Cortex*, 28(9), 3095-3114. https://doi.org/10.1093/cercor/bhx179
- Yeo, B. T. T., Krienen, F. M., Sepulcre, J., Sabuncu, M. R., Lashkari, D., Hollinshead, M., ... & Buckner, R. L. (2011). The organization of the human cerebral cortex estimated by intrinsic functional connectivity. *Journal of Neurophysiology*, 106(3), 1125-1165. https://doi.org/10.1152/jn.00338.2011
- ds000228 on OpenNeuro: https://openneuro.org/datasets/ds000228
- nilearn documentation: https://nilearn.github.io/stable/